# 📗 OpenAI API 활용 — 대화·파라미터·답변 생성 원리

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

지난 시간까지 우리는 문장을 **숫자 벡터(임베딩)** 로 바꾸고, 비슷한 문서를 **묶는(군집)** 데까지 왔습니다. 이제부터는 방향이 바뀝니다 — 이번 시간엔 **글을 이해하고 새 글을 만들어 내는** 언어모델(LLM)을 **OpenAI API** 로 직접 부려 봅니다. 챗봇에 말을 거는 것을 코드로 하는 셈입니다.

먼저 "모델이 도대체 **어떻게 답을 만들어 내는지**" 원리를 직관으로 익히고, 그 원리가 그대로 이어지는 **대화 API(Chat Completions·Responses)** 와 **파라미터(temperature·top_p·사고모드)** 를 다룹니다.

## ⏪ 복습 — 지난 시간까지

- **데이터 수집 단원**: `requests` 로 외부 **REST API** 를 호출해 JSON 을 받아 왔습니다. OpenAI API 도 같은 REST API 이고, 번거로운 부분을 `openai` 라이브러리가 대신 처리해 줄 뿐입니다.
- **텍스트 전처리 단원**: 문장을 **토큰**(의미 단위 조각)으로 쪼갰습니다. 언어모델도 글을 **토큰 단위**로 읽고 씁니다.
- **임베딩 단원**: `openai` 라이브러리로 **임베딩(embeddings)** 을 만들었습니다. 오늘은 같은 라이브러리의 **대화(chat)** 기능을 씁니다.

**오늘의 목표**

- [ ] 언어모델이 **다음 토큰을 확률로 예측**하며 답을 만든다는 원리를 설명한다.
- [ ] **Chat Completions API** 로 모델에 메시지를 보내고 답을 받는다(system·user·assistant 역할).
- [ ] **Responses API** 로 같은 일을 더 간단히 하고, 두 API 를 비교한다.
- [ ] **temperature·top_p·max_tokens** 로 답변의 무작위성과 길이를 조절한다.
- [ ] **사고모드(reasoning_effort)** 로 추론 모델이 얼마나 '깊게 생각'할지 조절한다.
- [ ] 배운 것을 묶어 **간단한 챗봇 함수**를 만든다.

아래 준비 셀을 먼저 실행하세요. **본인 API 키가 없어도 됩니다** — 키가 없으면 미리 저장해 둔 응답으로 실습이 그대로 진행됩니다(키가 있으면 실제 답을 받아 봅니다). 키를 넣는 방법은 폴더의 `.env.example` 를 참고하세요.

In [ ]:
# [제공 코드] OpenAI 클라이언트 준비 — 이 셀은 실행만 하세요.
# .env 에 OPENAI_API_KEY 가 있으면 실제 OpenAI 에 연결하고,
# 없으면 미리 저장해 둔 응답(data/api_cache.json)으로 진행됩니다(키·인터넷 없이 실습 가능).
# 아래 client 사용법은 공식 문서와 똑같습니다 → https://developers.openai.com/api/docs/guides/text
import sys
sys.path.insert(0, '.')          # openai_client.py 가 있는 폴더

from openai_client import get_client

client = get_client()

---
# 1. 모델은 어떻게 답을 만들까 — 다음 토큰 예측

## 왜 알아야 할까요?
곧 다룰 파라미터(temperature·top_p·사고모드)는 전부 "모델이 답을 만드는 **방식**"을 건드립니다. 그러니 원리를 한 번 잡아 두면 파라미터가 훨씬 쉬워집니다. (내부 수학은 과정 뒤쪽 딥러닝 단원에서 다루니, 여기서는 **직관**만 확실히 잡습니다.)

## 한 문장 원리
언어모델은 **다음에 올 토큰(글자 조각) 하나를 확률로 예측**하는 기계입니다. 그 한 토큰을 이어 붙이고, 늘어난 문장을 다시 넣어 **또 다음 토큰**을 예측합니다 — 이 과정을 끝날 때까지 **반복**합니다.

```
입력: "하늘은 파"
   모델이 계산한 다음 토큰 확률:   "랗다"(0.62)  "래서"(0.15)  "티"(0.08)  ...
   → 하나를 골라 이어 붙임 → "하늘은 파랗다"
   → 늘어난 문장을 다시 넣어 다음 토큰 예측 → ...  (문장이 끝날 때까지 반복)
```

## 트랜스포머는 여기서 무슨 일을 하나 (직관 한 줄)
확률을 잘 찍으려면 **문맥 전체**를 봐야 합니다. "강가에서 낚시로 잡은 __" 다음엔 '물고기', "은행에서 대출 받은 __" 다음엔 '돈' 이 와야 하죠. 요즘 언어모델의 뼈대인 **트랜스포머(Transformer)** 는 문장 안 단어들이 **서로 얼마나 관련 있는지(어텐션, attention)** 를 계산해, 문맥에 맞는 다음 토큰 확률을 정합니다. 우리는 이 확률분포에서 **어떻게 고를지**를 파라미터로 조절하게 됩니다.

<img src="images/self-attention.gif" width="720">

*어텐션의 실제 계산입니다. 입력마다 **query·key·value** 세 벡터를 만들고, 한 단어의 query 를 모든 단어의 key 와 곱해 **점수(score)** 를 냅니다. 그 점수로 value 들을 **가중합**한 것이 그 단어의 새 표현입니다 — "어느 단어를 얼마나 참고할지"가 점수로 정해지는 셈입니다.*

## 전체 구조는 어떻게 생겼나

<img src="images/transformer.png" width="900">

*트랜스포머는 **인코더**(왼쪽)와 **디코더**(오른쪽) 두 덩어리로 이뤄집니다. 둘 다 위 어텐션 블록을 N번 쌓은 것입니다. **BERT** 는 인코더만 써서 문장을 *이해*하고(11~13일차에 쓴 임베딩 모델이 이 계열), **GPT**(오늘 부르는 모델)는 디코더만 써서 다음 토큰을 *생성*합니다. 디코더의 **Masked** Multi-Head Attention 이 뒤쪽 단어를 못 보게 가려 주기 때문에 "앞만 보고 다음 한 글자를 예측"하는 것이 가능하고, 맨 위 **Softmax** 가 바로 그 다음 토큰의 확률분포입니다 — 4절에서 `temperature`·`top_p` 로 만지게 될 그 분포입니다.*


<img src="images/llm_autoregressive.gif" width="720">

*토큰 하나를 고르고 → 그 토큰을 문장 끝에 붙이고 → 늘어난 문장으로 다시 다음 토큰을 고른다. 이 되먹임을 **자기회귀(autoregressive) 생성**이라고 부릅니다.*

> 핵심: 답은 **정답표에서 찾아오는 것이 아니라**, 확률적으로 **한 토큰씩 생성**됩니다. 그래서 같은 질문에도 **매번 조금씩 다른 답**이 나올 수 있습니다(아래에서 직접 확인).

In [ ]:
# 같은 질문을 temperature=1 로 세 번 물어본다 — 확률적으로 생성되므로 답이 조금씩 달라진다
for i in range(3):
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': '가을을 한 문장으로 감성적으로 표현해줘.'}],
        temperature=1)
    print(f'{i+1}번째:', r.choices[0].message.content)

### 🖐️ 함께 따라하기 — 확률적 생성 직접 확인

같은 질문을 두 번 호출해 답이 서로 같은지/다른지 눈으로 확인해 봅니다(확률적 생성이라 달라질 수 있습니다).

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) '커피를 한마디로 표현하면?' 을 gpt-4o-mini 로 두 번 호출한다(각각 first, second)
# 2) 두 답(choices[0].message.content)을 각각 출력해 비교한다

### ✅ 바로 확인 퀴즈

**1.** 언어모델이 답을 만드는 방식을 한 문장으로 말하면?

<details><summary>정답 보기</summary>

**다음 토큰을 확률로 예측해 하나씩 이어 붙이는 것을 반복**합니다. 정답을 어딘가에서 찾아오는 게 아닙니다.

</details>

**2.** 같은 질문에 매번 똑같은 답이 나오지 않을 수 있는 이유는?

<details><summary>정답 보기</summary>

답이 **확률분포에서 골라져 생성**되기 때문입니다. 확률이 높은 토큰이 자주 뽑히지만, 다른 토큰이 뽑히면 문장이 달라집니다. 이 무작위성은 다음에 배울 **temperature** 로 조절합니다.

</details>

---
# 2. Chat Completions API — 모델과 대화하기

## 왜 필요할까요?
모델을 쓰려면 **메시지를 보내고 답을 받는** 통로가 필요합니다. 가장 널리 쓰이는 통로가 **Chat Completions** 입니다. 대화를 **메시지 목록**으로 표현합니다.

## 메시지의 세 가지 역할(role)
| 역할 | 뜻 | 예 |
|---|---|---|
| `system` | 모델에게 **역할·태도·규칙**을 지시(맨 앞) | "너는 친절한 한국어 비서다" |
| `user` | **사용자의 말**(질문·요청) | "파이썬이 뭐야?" |
| `assistant` | **모델의 이전 답변**(대화를 이어갈 때 넣음) | "파이썬은 프로그래밍 언어입니다" |

### 🎯 System Prompt — 셋 중 가장 중요한 하나
`system` 은 그냥 첫 메시지가 아니라 **모델의 성격·규칙을 고정하는 자리**입니다. 세 가지만 기억하세요.

- **언제 `system`, 언제 `user`?** — 매번 바뀌는 **그때그때의 요청**은 `user`, 대화 내내 변하지 않는 **역할·말투·출력 규칙**은 `system`. ("너는 번역기다. 영어로만 답한다" → `system` / "이 문장을 번역해줘" → `user`)
- **대화가 길어져도 유지된다** — `system` 은 맨 앞에 한 번만 넣으면 이후 모든 턴에 계속 적용됩니다. 매 턴 다시 넣을 필요가 없습니다(넣어도 되지만 토큰만 더 씁니다).
- **절대 규칙은 아니다** — 모델이 `system` 을 어길 수도 있습니다. 특히 `user` 가 정면으로 반대되는 요구를 하면 흔들립니다. 그래서 중요한 제약은 **코드에서 한 번 더 검증**해야 합니다(뒤에서 배울 **구조화된 출력**이 이 검증을 대신해 주는 좋은 예입니다).

## 문법
- **`client.chat.completions.create(model=..., messages=[...])`** — 모델과 메시지 목록을 넘긴다.
- 각 메시지는 **`{'role': 역할, 'content': 내용}`** 딕셔너리(앞서 배운 그 딕셔너리!).
- 답은 **`응답.choices[0].message.content`** 에 문자열로 들어 있다.
- **`응답.usage`** 로 이번 호출에 쓴 토큰 수(요금과 직결)를 볼 수 있다. **세 가지가 따로 있다:**
  - `usage.prompt_tokens` — 우리가 **보낸** 입력 토큰
  - `usage.completion_tokens` — 모델이 **생성한** 출력 토큰
  - `usage.total_tokens` — **입력 + 생성** 합계

> 뒤에서 배울 `max_tokens` 는 이 중 **생성 토큰만** 제한합니다. 그래서 `max_tokens=30` 으로 잘라도 `total_tokens` 는 30보다 큽니다(입력이 더해지니까) — 4절에서 직접 확인합니다.

In [ ]:
# 가장 기본적인 호출 — user 메시지 하나만 보낸다
resp = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': '파이썬을 초보자에게 한 문장으로 설명해줘.'}])

print('답변:', resp.choices[0].message.content)
print('쓴 토큰:', resp.usage.total_tokens)

In [ ]:
# system 으로 역할을 주면 답의 말투·형식이 달라진다
resp = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'system', 'content': '너는 이모지를 즐겨 쓰는 발랄한 말투의 도우미야.'},
        {'role': 'user', 'content': '파이썬을 초보자에게 한 문장으로 설명해줘.'}])
print(resp.choices[0].message.content)

### 대화 이어가기 — 앞의 답을 기억하게 하려면

모델은 **상태가 없습니다** — 매 호출을 처음 보는 것처럼 처리합니다. 그래서 대화를 이어가려면 이전 답을 다음 요청의 `messages` 에 **다시 넣어야** 합니다. 모델의 이전 답변은 **`assistant`** 역할로 넣습니다.

In [ ]:
# 1차 질문 → 답을 받는다
messages = [{'role': 'user', 'content': '간단한 파이썬 팁 하나만 알려줘.'}]
reply1 = client.chat.completions.create(model='gpt-4o-mini', messages=messages).choices[0].message.content
print('1차:', reply1)

# 모델의 답을 assistant 로 messages 에 이어 붙이고, 후속 질문을 한다
messages.append({'role': 'assistant', 'content': reply1})
messages.append({'role': 'user', 'content': '방금 그 팁을 초등학생도 알게 더 쉽게 설명해줘.'})
reply2 = client.chat.completions.create(model='gpt-4o-mini', messages=messages).choices[0].message.content
print('2차:', reply2)

> `assistant` 메시지를 빼먹으면 모델은 '방금 그 팁'이 무엇인지 알지 못합니다. 이렇게 **이력을 쌓아 다시 보내는 것**이 멀티턴 대화의 원리입니다.

### 스트리밍 — 답을 다 만들기 전에 먼저 보여 주기

긴 답은 완성될 때까지 몇 초가 걸립니다. 그동안 화면이 멈춰 있으면 사용자는 고장으로 느낍니다. **`stream=True`** 를 주면 답이 **조각(chunk)** 으로 오는 대로 화면에 흘려보낼 수 있습니다 — ChatGPT 에서 글자가 타이핑되듯 나오는 그 방식입니다.

## 문법
- **`client.chat.completions.create(..., stream=True)`** → 응답 대신 **조각들의 반복자**가 온다.
- 조각의 새 글자는 **`chunk.choices[0].delta.content`** 에 들어 있다(없는 조각도 있으므로 `if` 로 거른다).
- `print(..., end='', flush=True)` 로 줄바꿈 없이 이어 찍는다.

> 총 걸리는 시간이 줄어드는 것은 아닙니다. **첫 글자가 보이기까지의 시간**이 줄어 체감이 달라지는 것입니다. 대신 답이 다 오기 전엔 전체 문자열을 쓸 수 없어서, **후처리(집계·파싱)가 필요한 작업에는 맞지 않습니다.**

In [ ]:
# stream=True — 답이 조각으로 오는 대로 이어서 출력한다
stream = client.chat.completions.create(model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': '파이썬으로 할 수 있는 일을 세 문장으로 설명해줘.'}],
    stream=True)

pieces = []
for chunk in stream:
    piece = chunk.choices[0].delta.content
    if piece:                       # 내용이 없는 조각도 섞여 온다
        print(piece, end='', flush=True)
        pieces.append(piece)

full = ''.join(pieces)              # 조각을 모으면 평소의 그 답 문자열이 된다
print('\n\n조각 수:', len(pieces), '| 전체 길이:', len(full))

### 🖐️ 함께 따라하기 — 번역기처럼 시켜 보기

`system` 메시지로 "한국어를 영어로 번역만 하라"고 지시하고, `user` 로 한국어 문장을 보내 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) messages 를 만든다: system='입력한 한국어를 영어로 번역만 해. 설명은 하지 마.',
#    user='오늘 날씨가 정말 좋네요.'
# 2) client.chat.completions.create(model='gpt-4o-mini', messages=...) 를 호출한다
# 3) 답변 문자열을 출력한다

### ✅ 바로 확인 퀴즈

**1.** 모델에게 "너는 ~한 역할이다"라고 태도·규칙을 지시하는 메시지 역할(role)은 무엇인가요?

<details><summary>정답 보기</summary>

**`system`** 입니다. 보통 메시지 목록의 맨 앞에 한 번 넣어 모델의 역할·말투·규칙을 정합니다.

</details>

**2.** 응답에서 모델이 실제로 한 말(문자열)은 어디에 들어 있나요?

<details><summary>정답 보기</summary>

**`resp.choices[0].message.content`** 에 있습니다. `usage` 에는 토큰 사용량이 들어 있습니다.

</details>

---
## 어떤 모델을 쓸까 — 주요 모델과 비용

위에서 `model='gpt-4o-mini'` 라고 적었습니다. 모델은 **작업에 맞춰 고르는 것**이고, 고르는 기준은 **품질·속도·비용** 세 가지입니다. 아래는 공식 가격표·모델 문서에서 그대로 옮긴 값입니다(**2026년 8월 기준 — 가격과 라인업은 자주 바뀌므로 반드시 공식 페이지에서 최신 값을 확인하세요**).

| 모델 | 입력 $/1M토큰 | 출력 $/1M토큰 | 컨텍스트 | 추론 | 어떤 작업에 |
|---|---|---|---|---|---|
| `gpt-5.6-sol` | 5.00 | 30.00 | 1.05M | ✅ | 복잡한 전문 작업(최상위) |
| `gpt-5.6-terra` | 2.00 | 12.00 | 1.05M | ✅ | 지능과 비용의 균형 |
| `gpt-5.6-luna` | 0.20 | 1.20 | 1.05M | ✅ | 대량·비용 민감 작업 |
| `gpt-5` | 1.25 | 10.00 | 400K | ✅ | 이전 세대 상위 모델 |
| `gpt-5-mini` | 0.25 | 2.00 | 400K | ✅ | 잘 정의된 작업·저지연 대량 처리 |
| `gpt-5-nano` | 0.05 | 0.40 | 400K | ✅ | GPT-5 중 가장 싸고 빠름(요약·분류) |
| `gpt-4o` | 2.50 | 10.00 | 128K | ❌ | 이전 세대 범용 |
| **`gpt-4o-mini`** | **0.15** | **0.60** | 128K | ❌ | **오늘 실습에 쓰는 모델** |

**컨텍스트**는 한 번에 넣을 수 있는 최대 토큰(대화 + 문서), **추론**은 답하기 전에 속으로 생각하는 모델인지입니다(5절에서 다룹니다).

### 왜 이 수업은 `gpt-4o-mini` 를 쓸까 — 이유 두 가지

**① 추론 모델은 이번 단원의 파라미터를 받지 않습니다.** 실제로 `gpt-5-nano` 에 넣어 보면 이렇게 거부됩니다.

```
temperature=1.2  → 400  Unsupported value: 'temperature' does not support 1.2 with this model.
top_p=0.1        → 400  Unsupported parameter: 'top_p' is not supported with this model.
max_tokens=25    → 400  Unsupported parameter: 'max_tokens' ... Use 'max_completion_tokens' instead.
```

4절에서 배울 `temperature`·`top_p`·`max_tokens` 를 실습하려면 **비추론 모델**이어야 합니다.

**② 토큰 단가가 싸다고 요금이 싼 것이 아닙니다.** 같은 작업(선크림 리뷰 5건 구조화 감정분석)을 세 모델로 실제 호출해 청구 토큰을 잰 결과입니다.

<img src="images/생각토큰_비용.jpg" width="820">

*왼쪽: 비추론 모델은 바로 답한다(동전 하나). 오른쪽: 추론 모델은 **같은 답**을 내놓기 전에 속으로 길게 생각하고, 그 생각이 전부 요금으로 쌓인다(동전 더미). **말풍선(답)의 크기는 둘이 같다** — 결과는 같은데 값만 다르다는 뜻이다.*

| 모델 | 입력 토큰 | 출력 토큰(그중 생각) | 5건 요금 | 판정 결과 |
|---|---|---|---|---|
| `gpt-4o-mini` | 1,254 | 292 (0) | **$0.000363** | 부·부·부·긍·긍 |
| `gpt-5-nano` | 1,249 | 11,899 (**11,328**) | $0.004822 (**13.3배**) | 같음 |
| `gpt-5.6-luna` | 1,254 | 448 (30) | $0.000788 (2.2배) | 같음 |

`gpt-5-nano` 는 토큰 단가가 `gpt-4o-mini` 보다 **싼데도**(입력 0.05 vs 0.15) 실제 요금은 **13배**였습니다. **추론 모델이 답하기 전에 쓴 '생각 토큰' 11,328개가 전부 출력 요금으로 청구**되기 때문입니다. 게다가 세 모델의 감정 판정은 **똑같았습니다** — 이 정도 난이도의 작업에 추론 모델을 쓰면 돈만 더 내는 셈입니다.

> **고르는 기준 한 줄**: 단가표가 아니라 **그 작업에서 실제로 나가는 토큰**으로 비교하세요. 간단한 분류·추출은 작은 비추론 모델, 여러 단계를 밟아야 하는 문제는 추론 모델입니다.

### 📚 공식 문서
- **가격표** — https://developers.openai.com/api/docs/pricing
- **모델 목록·사양** — https://developers.openai.com/api/docs/models
- **텍스트 생성 가이드(Chat Completions·Responses)** — https://developers.openai.com/api/docs/guides/text
- **Chat Completions API 레퍼런스** — https://developers.openai.com/api/docs/api-reference/chat

---
# 3. Responses API — 더 간단한 새 통로

## 왜 필요할까요?
OpenAI 는 최근에 더 간결한 **Responses API** 를 내놨습니다. 메시지 목록 대신 **`input`** 에 문자열(또는 메시지)을 바로 주고, 답도 **`output_text`** 한 줄로 꺼냅니다. 하는 일은 Chat Completions 와 같지만 손이 덜 갑니다.

## 문법 — Chat Completions 와 비교
| | Chat Completions | Responses |
|---|---|---|
| 호출 | `client.chat.completions.create` | `client.responses.create` |
| 입력 | `messages=[{'role','content'}, ...]` | `input='문자열'` (또는 messages) |
| 답 꺼내기 | `resp.choices[0].message.content` | `resp.output_text` |

> 둘 다 알아 두면 됩니다. 예전 코드·자료는 Chat Completions 가 많고, 새로 만들 땐 Responses 가 간편합니다. 오늘 배우는 파라미터(temperature 등)는 **양쪽 모두** 비슷하게 씁니다.

In [ ]:
# Responses API — input 에 질문 문자열을 바로 준다
resp = client.responses.create(
    model='gpt-4o-mini',
    input='세종대왕을 한 문장으로 소개해줘.')
print(resp.output_text)

### 🖐️ 함께 따라하기 — Responses 로 물어보기

Responses API 로 "파이썬과 자바의 차이를 한 문장으로" 물어봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) client.responses.create(model='gpt-4o-mini', input='파이썬과 자바의 차이를 한 문장으로 알려줘.') 호출
# 2) 결과의 output_text 를 출력한다

### ✅ 바로 확인 퀴즈

**1.** Responses API 에서 모델의 답 문자열을 꺼내는 속성은 무엇인가요?

<details><summary>정답 보기</summary>

**`resp.output_text`** 입니다. Chat Completions 의 `resp.choices[0].message.content` 에 해당합니다.

</details>

**2.** Responses API 에서 질문을 넘기는 인자 이름은?

<details><summary>정답 보기</summary>

**`input`** 입니다. 문자열을 바로 주거나, Chat 처럼 메시지 목록을 줄 수도 있습니다.

</details>

---
# 4. 파라미터로 답을 조절하기 — temperature·top_p·max_tokens

## 왜 필요할까요?
1절에서 답은 **확률분포에서 골라 생성**된다고 했습니다. 그 "고르는 방식"을 파라미터로 조절합니다. 번역·요약처럼 **일관된** 답이 필요할 때와, 아이디어·카피처럼 **다양한** 답이 필요할 때를 다르게 쓸 수 있습니다.

## 세 가지 핵심 파라미터
| 파라미터 | 범위 | 효과 | 언제 |
|---|---|---|---|
| `temperature` | 0 ~ 2 | 낮으면 **일관**(확률 높은 토큰 위주), 높으면 **다양·창의** | 0~0.3 사실·번역 / 0.8~1.2 창작 |
| `top_p` | 0 ~ 1 | 확률 상위 몇 %의 토큰만 후보로. 낮으면 안전·좁게 | temperature 대신 쓰기도 함 |
| `max_tokens` | 정수 | **답의 최대 길이**(토큰 수). 요금·속도 제어 | 답이 너무 길 때 |

> 비유: `temperature` 는 확률분포를 **뾰족하게(낮음)** 또는 **평평하게(높음)** 만드는 파라미터입니다. 평평할수록 덜 유력한 토큰도 자주 뽑혀 답이 다양해집니다. `temperature` 와 `top_p` 는 보통 **둘 중 하나만** 조절합니다.

> **그럼 둘 중 무엇을 만질까요?** — 실무 기본값은 **`temperature` 를 먼저** 건드리는 것입니다. 직관적이고(일관 ↔ 창의) 대부분의 조절이 이걸로 끝납니다. **`top_p` 는 "이상한 단어가 가끔 튀어나오는 것" 자체를 막고 싶을 때** 씁니다 — 확률이 낮은 꼬리 후보를 아예 후보에서 빼 버리기 때문입니다. 둘을 동시에 낮추면 후보가 이중으로 좁아져 답이 지나치게 뻔해지므로 한쪽만 씁니다.

<img src="images/temperature_top_p.png" width="960">

그림의 왼쪽 세 칸은 **같은 분포**를 temperature 로 다시 빚은 모습이고, 오른쪽 한 칸은 **top_p 가 후보를 몇 개까지 남기는지**입니다. 아래에서 이 계산을 직접 해 봅니다 — 모델 안에서 일어나는 일과 같은 방식입니다.

In [ ]:
# 1절의 그 분포로, temperature 가 확률을 어떻게 다시 빚는지 직접 계산해 본다
import numpy as np

tokens = ['랗다', '래서', '티', '스텔', '랑']
probs = np.array([0.62, 0.15, 0.08, 0.06, 0.05])
probs = probs / probs.sum()

def reshape(p, temperature):
    logits = np.log(p) / temperature       # 온도로 나눈다
    e = np.exp(logits - logits.max())
    return e / e.sum()                     # 다시 확률로

for t in [0.2, 1.0, 1.5]:
    r = reshape(probs, t)
    print(f'T={t:<4}', '  '.join(f'{tok} {v:.2f}' for tok, v in zip(tokens, r)))
print('→ T 가 낮으면 1등에 몰리고(늘 같은 답), 높으면 고르게 퍼진다(답이 다양)')

In [ ]:
# top_p 는 확률을 큰 것부터 더해 가다 기준을 넘는 순간 끊고, 남은 후보 중에서만 고른다
p = reshape(probs, 1.0)
order = np.argsort(-p)                      # 확률 큰 순서
cum = np.cumsum(p[order])                   # 누적 확률

for tp in [0.5, 0.9, 1.0]:
    keep = int(np.searchsorted(cum, tp)) + 1
    print(f'top_p={tp}', '→ 후보', keep, '개:', [tokens[i] for i in order[:keep]])
print('→ top_p 가 낮을수록 후보가 좁아져 안전한 답만 나온다')

In [ ]:
# 실제 API 에도 그대로 넘긴다 (temperature 와 top_p 는 보통 둘 중 하나만)
# 같은 요청을 세 번씩 — 후보가 좁으면(0.1) 답이 거의 그대로, 넓으면(1.0) 매번 달라진다
prompt_p = '가을에 어울리는 카페 이름 하나만 지어줘. 이름만 답해.'

for tp in [0.1, 1.0]:
    names = []
    for _ in range(3):
        resp = client.chat.completions.create(model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt_p}],
            top_p=tp)
        names.append(resp.choices[0].message.content.strip())
    print(f'top_p={tp} → {names}')

print('\n→ top_p 가 낮으면 확률 높은 후보만 남아 매번 비슷한 답, 높으면 꼬리 후보까지 살아나 답이 갈린다')

In [ ]:
# temperature 0 vs 1.2 — 같은 창작 요청에 답의 다양성이 어떻게 달라지나
prompt = '새 커피 브랜드 이름을 하나만 지어줘. 이름만 답해.'

cold = client.chat.completions.create(model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': prompt}], temperature=0)
hot = client.chat.completions.create(model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': prompt}], temperature=1.2)

print('temperature=0   →', cold.choices[0].message.content)
print('temperature=1.2 →', hot.choices[0].message.content)

In [ ]:
# max_tokens 로 답 길이를 제한한다 (짧게 끊긴다)
resp = client.chat.completions.create(model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': '인공지능의 역사를 설명해줘.'}],
    max_tokens=30)
print(resp.choices[0].message.content)

u = resp.usage
print('\n생성 토큰(completion):', u.completion_tokens, '  ← max_tokens=30 이 자른 것은 이 값이다')
print('입력 토큰(prompt)   :', u.prompt_tokens)
print('합계(total)         :', u.total_tokens, ' = 입력 + 생성')

### 🖐️ 함께 따라하기 — 사실 질문엔 temperature=0

번역·사실 확인처럼 **일관된 답**이 필요한 질문은 `temperature=0` 으로 부릅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) '대한민국의 수도는? 도시 이름만 답해.' 를 temperature=0 으로 호출한다
# 2) 답을 출력한다

### ✅ 바로 확인 퀴즈

**1.** 번역·요약처럼 **일관된** 답이 필요할 때 temperature 는 높게, 낮게 중 어느 쪽이 좋을까요?

<details><summary>정답 보기</summary>

**낮게(0에 가깝게)** 씁니다. 확률이 높은 토큰 위주로 골라 매번 비슷하고 안정적인 답을 냅니다.

</details>

**2.** 답이 너무 길어 요금·속도가 부담일 때 쓰는 파라미터는?

<details><summary>정답 보기</summary>

**`max_tokens`** 입니다. 생성할 답의 최대 토큰 수를 제한합니다.

</details>

---
# 5. 사고모드 — 얼마나 깊게 생각하게 할까 (reasoning_effort)

## 왜 필요할까요?
요즘은 답하기 전에 **속으로 단계를 밟아 생각(추론)** 하는 **추론 모델**(예: `gpt-5-mini`)이 있습니다. 복잡한 수학·논리 문제엔 깊게 생각할수록 정확하지만, **생각도 토큰을 쓰므로** 느리고 비쌉니다. **`reasoning_effort`** 로 그 깊이를 `'low'·'medium'·'high'` 로 조절합니다.

## 문법
- **`client.chat.completions.create(model='gpt-5-mini', messages=[...], reasoning_effort='low')`**
- 생각에 쓴 토큰은 **`resp.usage.completion_tokens_details.reasoning_tokens`** 로 볼 수 있습니다.

| effort | 생각 깊이 | 속도·비용 | 언제 |
|---|---|---|---|
| `low` | 얕게 | 빠르고 쌈 | 간단한 질문 |
| `medium` | 보통 | 중간 | 일반적 |
| `high` | 깊게 | 느리고 비쌈 | 어려운 수학·논리·코딩 |

> ⚠️ **길이 제한 인자 이름이 다릅니다.** 4절에서 쓴 `max_tokens` 를 추론 모델에 그대로 주면 거부됩니다 (`400 Unsupported parameter: 'max_tokens' is not supported with this model. Use 'max_completion_tokens' instead.`). 추론 모델에는 **`max_completion_tokens`** 를 씁니다.

> 그리고 이 상한은 **생각 토큰까지 포함**합니다. 너무 작게 주면 생각만 하다 끝나 **답이 빈 문자열**로 옵니다 — 실측: `max_completion_tokens=16` → `content=''`, `reasoning_tokens=16`, `finish_reason='length'`. 같은 질문에 `2000` 을 주면 `'5600'` 이 정상적으로 옵니다.

> 쉬운 질문에 `high` 를 쓰면 돈·시간만 낭비됩니다. **문제 난이도에 맞춰** 고르는 것이 핵심입니다.

> 아래 시연은 **일부러 쉬운 산수**를 씁니다. 낭비를 권하는 게 아니라, **쉬운 문제에서는 `high` 가 무엇을 더 주는지(그리고 안 주는지)** 를 눈으로 보기 위해서입니다.

In [ ]:
# 같은 문제를 low 와 high 로 각각 풀어, 생각에 쓴 토큰(reasoning_tokens)을 비교한다
problem = '사과 3개가 2400원이면 7개는 얼마인가요? 숫자만 답해주세요.'
for effort in ['low', 'high']:
    resp = client.chat.completions.create(model='gpt-5-mini',
        messages=[{'role': 'user', 'content': problem}],
        reasoning_effort=effort)
    rt = resp.usage.completion_tokens_details.reasoning_tokens
    print(f'[{effort}] 답: {resp.choices[0].message.content}  | 생각에 쓴 토큰: {rt}')
print('\n→ 답은 둘 다 같다. 이 정도 문제엔 high 가 정확도를 더 주지 못하고 생각 토큰만 몇 배로 쓴다.')
print('   high 가 값을 하는 건 여러 단계를 거쳐야 풀리는 어려운 문제다 — 난이도에 맞춰 고르는 이유.')

### 🖐️ 함께 따라하기 — 난이도에 맞는 effort 고르기

간단한 상식 질문은 `reasoning_effort='low'` 로 부르면 충분합니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) model='gpt-5-mini', reasoning_effort='low' 로
#    '일주일은 며칠인가요? 숫자만.' 을 물어본다
# 2) 답을 출력한다

### ✅ 바로 확인 퀴즈

**1.** 아주 어려운 논리·수학 문제에는 `reasoning_effort` 를 어떻게 두는 게 좋을까요?

<details><summary>정답 보기</summary>

**`'high'`** 로 둡니다. 깊게 생각할수록 정확해지지만 느리고 비싸지므로, 어려운 문제에만 씁니다.

</details>

**2.** 모델이 답하기 전에 '속으로 생각'하는 데 쓴 토큰은 어디서 확인하나요?

<details><summary>정답 보기</summary>

**`resp.usage.completion_tokens_details.reasoning_tokens`** 입니다. 생각도 토큰을 쓰므로 요금에 포함됩니다.

</details>

**3.** 추론 모델(`gpt-5-mini`)의 답 길이를 제한하려고 4절에서 배운 `max_tokens` 를 그대로 주면 어떻게 될까요?

<details><summary>정답 보기</summary>

**거부됩니다**(`400 Unsupported parameter`). 추론 모델에는 **`max_completion_tokens`** 를 씁니다. 게다가 그 상한은 **생각 토큰까지 포함**하므로 너무 작게 주면 생각만 하다 끝나 **답이 빈 문자열**로 옵니다.

</details>

---
## 🚀 응용 클론코딩 — 나만의 간단 챗봇 함수

지금까지 배운 것을 하나로 묶어, **역할(system)과 온도를 정해 질문에 답하는 함수**를 만들어 봅니다. 아래 지시대로 직접 완성해 보세요.

In [ ]:
# 🚀 응용 (아래 순서대로 직접 작성해 보세요)
# 1) ask(question, persona='너는 친절한 한국어 도우미야.', temperature=0.7) 함수를 정의한다
# 2) 함수 안에서 messages 를 [system=persona, user=question] 으로 만든다
# 3) client.chat.completions.create(model='gpt-4o-mini', messages=..., temperature=temperature) 호출
# 4) 답변 문자열(content)을 return 한다
# 5) ask('추천 취미 하나만 알려줘') 와
#    ask('추천 취미 하나만 알려줘', persona='너는 무뚝뚝한 해적이야.') 를 각각 출력해 말투 차이를 본다

---
## 이번 강의 정리

| 주제 | 핵심 | 코드 |
|---|---|---|
| 답변 생성 원리 | 다음 토큰을 확률로 예측·반복 생성 | — |
| Chat Completions | 메시지 목록(system·user·assistant)으로 대화 | `client.chat.completions.create` |
| Responses | 더 간단한 새 통로 | `client.responses.create` / `output_text` |
| temperature·top_p | 답의 무작위성(일관 ↔ 다양) | `temperature=0`~`1.2` |
| max_tokens | 답 길이 제한 | `max_tokens=...` |
| 사고모드 | 추론 깊이·비용 조절 | `reasoning_effort='low/medium/high'` |

- 모델의 답은 **확률적으로 생성**되므로 파라미터로 **성격**을 정할 수 있습니다.
- 사실·번역엔 **낮은 temperature**, 창작엔 **높은 temperature**, 어려운 추론엔 **높은 effort**.

## ⏭️ 예고 — 다음 시간

이번 시간엔 모델과 **대화**하고 **파라미터**로 답을 조절했습니다. 다음 시간엔 모델을 더 똑똑하게 부리는 법 — **프롬프트 엔지니어링**으로 지시를 잘 쓰고, **Function Calling** 으로 모델이 직접 도구(함수)를 부르게 하고, **구조화된 출력**으로 답을 **딱 정해진 형식(JSON)** 으로 받아 **리뷰 감정분석**까지 해 봅니다.

수고하셨습니다!